In [6]:

import numpy as np
import networkx as nx
from scipy.stats import qmc
import EoN
from pathlib import Path
import pickle
from scipy.stats import qmc
from pathlib import Path
from tqdm import tqdm
from pathlib import Path
import csv
import pymc as pm 
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("experiments/mcmc-sampling/data/raw")
PLOTS_DIR = Path("experiments/mcmc-sampling/out/plots/mcmc_sampling_plots")
MCMC_DIR = Path("experiments/mcmc-sampling/data/raw/mcmc_posterior_results")
OUTPUT_PKL = DATA_DIR /"abm-data.pkl"
OUTPUT_CSV = DATA_DIR /"abm-data.csv"

# PARAMETERS
N = 100000              # network size
m = 10                      # Barabasi–Albert attachment parameter

tmax=200
n_timepoints=200
initial_samples=100  # initial Sobol samples #500
sigma = 1.0      # width of R0 target distribution
n_replicates=1  # replicates of parameter sets 

PARAM_RANGES = {
    'tau':(0.0005,0.024),
    'gamma':(0.007,0.5),
    'rho':(0.001,0.01)
}

PARAM_NAMES = ['tau', 'gamma', 'rho']
output_path = Path('abm-data.pkl')


ratio=34.0 # calculated for 100000 graphs

net_stats = {
    'k_avg': 9.9988,
    'k2_avg': 266.020,
    'ratio': ratio,
    'k_std': 1.5,
    'k_max': 20
}



# R0 COMPUTATION
def compute_R0(samples,ratio):
    """
    Compute epidemic reproduction number.

    R0 = (tau/gamma) * (<k²> - <k>) / <k>
    """

    tau = samples[:, 0]
    gamma = samples[:, 1]

    R0 = (tau/gamma) * ratio  

    return R0

# Using MCMC to sample from the target distribution  with Uniform distribution

with pm.Model() as model:
    
    # Priors: Uniform over plausible ranges
    tau   = pm.Uniform("tau", lower=PARAM_RANGES['tau'][0], upper=PARAM_RANGES['tau'][1])
    gamma = pm.Uniform("gamma", lower=PARAM_RANGES['gamma'][0], upper=PARAM_RANGES['gamma'][1])
    rho   = pm.Uniform("rho", lower=PARAM_RANGES['rho'][0], upper=PARAM_RANGES['rho'][1])

    # Compute R0 deterministically
    R0 = pm.Deterministic("R0", (tau / gamma) * ratio)
    
    # Target density: Gaussian around R0 ≈ 1
    logp = -0.5 * ((R0 - 1.0) / sigma) ** 2
    pm.Potential("R0_target", logp)
    
    # Sample with NUTS
   
    trace = pm.sample(  
        draws=initial_samples,
        tune=3000, 
        chains=4 ,  # thus Total_sample=samples*chains
        cores=1,
        thin=10,
        target_accept=0.95,
        random_seed=43
    )
#Thin after sampling.
trace_thinned = trace.sel(draw=slice(None, None, 10))

# Convert trace to (n_samples, 3) array: tau, gamma, rho
posterior_samples=np.vstack([
    trace_thinned.posterior['tau'].values.flatten(),
    trace_thinned.posterior['gamma'].values.flatten(),
    trace_thinned.posterior['rho'].values.flatten()
]).T


# print(az.summary(trace_thinned, var_names=["tau", "gamma", "rho"]))
# summary_df = az.summary(trace_thinned, var_names=["tau", "gamma", "rho"])
# summary_df.to_csv(MCMC_DIR/ "mcmc_summary.csv")

# az.plot_trace(
#     trace,
#     var_names=["tau", "gamma", "rho"],
#     compact=False,
#     figsize=(12, 8)
# )

# plt.tight_layout()
# trace_plot_path = PLOTS_DIR / "mcmc_trace_plot.png"
# plt.savefig(trace_plot_path, dpi=200, bbox_inches='tight')
# plt.show()
# print(f"Saved: {trace_plot_path}")


#posterior sample dimensions 
print(posterior_samples.shape) #1000,3


def run_batch(G, posterior_samples, n_replicates, tmax, n_timepoints, seed=None):
    
    if seed is not None:
        np.random.seed(seed)

    t_fixed = np.linspace(0, tmax, n_timepoints)
    all_sims = []

    for tau, gamma, rho in tqdm(posterior_samples, desc="Running batch simulations"):

        for rep in range(n_replicates):

            t, S, I, R = EoN.fast_SIR(G, tau, gamma, rho=rho, tmax=tmax)

            all_sims.append({
                'params': {
                    'tau': float(tau),
                    'gamma': float(gamma),
                    'rho': float(rho),
                },
                'output': {
                    't': t_fixed,
                    'S': np.interp(t_fixed, t, S),
                    'I': np.interp(t_fixed, t, I),
                    'R': np.interp(t_fixed, t, R),
                },
                'replicate_id': rep,
            })

    print(f"Generated {len(all_sims)} simulations "
          f"({len(posterior_samples)} param sets × {n_replicates} replicates)")

    return all_sims



# Build dataset structure

def build_dataset(all_sims, G, net_stats, m, n_replicates, param_ranges):

    n_timepoints = len(all_sims[0]['output']['t'])

    dataset = {

        'simulations': all_sims,

        'network': {
            'type': 'barabasi_albert',
            'N': G.number_of_nodes(),
            'm': m,
            'ratio': net_stats['ratio'],
            'graph': G,
        },

        'metadata': {
            'n_samples': len(all_sims),
            'n_replicates': n_replicates,
            'n_timepoints': n_timepoints,
            'param_ranges': param_ranges,
            'R0_formula': 'R0 = (tau/gamma) * <k²>/<k>',
            'sampling_strategy': 'MCMC',
        }
    }

    return dataset

# Aggregate simulations for plotting
def summarise_for_plot(dataset):

    sims = dataset['simulations']
    t_fixed = sims[0]['output']['t']

    all_I = np.array([s['output']['I'] for s in sims])
    all_S = np.array([s['output']['S'] for s in sims])
    all_R = np.array([s['output']['R'] for s in sims])

    return {
        't': t_fixed,
        'all_I': all_I,
        'all_S': all_S,
        'all_R': all_R,
        'I_mean': all_I.mean(axis=0),
        'I_p10': np.percentile(all_I, 10, axis=0),
        'I_p25': np.percentile(all_I, 25, axis=0),
        'I_p75': np.percentile(all_I, 75, axis=0),
        'I_p90': np.percentile(all_I, 90, axis=0),
    }


# Plot epidemic uncertainty
def plot_sir_uncertainty(full_results,N, output_dir=PLOTS_DIR):

    t = full_results['t']
    all_I = full_results['all_I']
    all_R = full_results['all_R']

    final_R = all_R[:, -1]

    extinct_mask = final_R < 0.01 * N
    outbreak_mask = ~extinct_mask

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fig.suptitle(
        "SIR Epidemic Trajectories — Posterior + Stochastic Uncertainty",
        fontsize=13,
        fontweight='bold'
    )

    for ax, mask, label, color in [

        (axes[0], extinct_mask, "Extinction / near-threshold", "steelblue"),
        (axes[1], outbreak_mask, "Outbreak", "firebrick"),

    ]:

        subset = all_I[mask]
        n = mask.sum()

        if n == 0:
            ax.set_title(f"{label}\n(no trajectories)")
            continue

        mean_I = subset.mean(axis=0)
        p10 = np.percentile(subset, 10, axis=0)
        p25 = np.percentile(subset, 25, axis=0)
        p75 = np.percentile(subset, 75, axis=0)
        p90 = np.percentile(subset, 90, axis=0)

        ax.fill_between(t, p10, p90, color=color, alpha=0.15)
        ax.fill_between(t, p25, p75, color=color, alpha=0.30)

        ax.plot(t, mean_I, color=color, linewidth=2)

        ax.set_xlabel("Time")
        ax.set_ylabel("Number infectious")
        ax.set_title(f"{label} (n={n})")
        ax.set_ylim(bottom=0)

        ax.grid(True, alpha=0.3)

    output_dir = Path(output_dir)
    out_path   = output_dir / "trajectories_split.png"
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out_path}")



# Save dataset (pickle)
def save_dataset(dataset, filepath):

    with open(filepath, 'wb') as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Dataset saved {filepath} ({len(dataset['simulations'])} sims)")



# Save summary CSV


def save_csv(dataset, filepath):

    sims = dataset['simulations']
    ratio = dataset['network']['ratio']
    N = dataset['network']['N']

    fields = [
        'sim_id', 'replicate_id',
        'tau', 'gamma', 'rho',
        'R0', 'peak_I', 'peak_time',
        'final_R', 'attack_rate', 'near_threshold'
    ]

    with open(filepath, 'w', newline='') as f:

        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()

        for sim_id, sim in enumerate(sims):

            tau = sim['params']['tau']
            gamma = sim['params']['gamma']
            rho = sim['params']['rho']

            R0 = (tau / gamma) * ratio

            I = sim['output']['I']
            R = sim['output']['R']
            t = sim['output']['t']

            peak_I = float(I.max())

            peak_time = float(t[I.argmax()]) if peak_I > 0 else np.nan

            writer.writerow({

                'sim_id': sim_id,
                'replicate_id': sim.get('replicate_id', 0),

                'tau': tau,
                'gamma': gamma,
                'rho': rho,

                'R0': round(R0, 4),

                'peak_I': peak_I,
                'peak_time': peak_time,

                'final_R': float(R[-1]),
                'attack_rate': float(R[-1] / N),

                'near_threshold': int(abs(R0 - 1) < 0.2),
            })

    print(f"CSV saved to {filepath} ({len(sims)} rows)")

# Entry point
G = nx.barabasi_albert_graph(N, m)

all_sims = run_batch(
    G,
    posterior_samples,
    n_replicates,
    tmax,
    n_timepoints,
)

dataset = build_dataset(
    all_sims,
    G,
    net_stats,
    m,
    n_replicates,
    PARAM_RANGES,
)

full_results = summarise_for_plot(dataset)

# # Plots → PLOTS_DIR 
# plot_sir_uncertainty(full_results, N=G.number_of_nodes(), output_dir=PLOTS_DIR)

# Data → DATA_DIR 
save_dataset(dataset, OUTPUT_PKL)   
save_csv(dataset, OUTPUT_CSV)       

# Exploration 
data = pd.read_csv(OUTPUT_CSV)      

print(data.columns)
print(data.head(20))
print(f"\nTotal rows   : {len(data)}")
print(f"Missing values:\n{data.isnull().sum()}")
print(f"Shape        : {data.shape}")
print(f"\n{data.describe(include='all')}")

total_samples = len(data)
greater  = data[data['R0'] > 1.2]
between  = data[(data['R0'] >= 0.5) & (data['R0'] <= 1.5)]
less_than = data[data['R0'] < 0.8]

print(f"\nR₀ > 1.5     : {len(greater)}  ({len(greater)/total_samples*100:.1f}%)")
print(f"0.5 ≤ R₀ ≤ 1.5: {len(between)} ({len(between)/total_samples*100:.1f}%)")
print(f"R₀ < 0.5      : {len(less_than)}  ({len(less_than)/total_samples*100:.1f}%)")

print(f"\nData : {DATA_DIR}")
print(f"Plots : {PLOTS_DIR}")


# #Saves to PLOTS_DIR
# slope = 1/ratio
# plt.figure(figsize=(10, 10))
# plt.scatter(data['gamma'], data['tau'], alpha=0.2)

# x_vals = np.linspace(data['gamma'].min(), data['gamma'].max(), 100)
# y_vals = slope * x_vals
# plt.plot(x_vals, y_vals, color='red', linestyle='--', label=f'slope={slope:.5f}')

# plt.xlabel('gamma')
# plt.ylabel('tau')
# plt.title('Scatter plot of tau vs gamma — MCMC Posterior Samples')
# plt.legend()
# plt.grid(True)

# scatter_path = PLOTS_DIR / "tau_gamma_scatter.png"
# plt.savefig(scatter_path, dpi=200, bbox_inches='tight')
# plt.show()
# print(f"Saved: {scatter_path}")





Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [tau, gamma, rho]


Sampling 3 chains for 3_000 tune and 100 draw iterations (9_000 + 300 draws total) took 42 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


(30, 3)


Running batch simulations: 100%|██████████| 30/30 [00:07<00:00,  3.98it/s]

Generated 30 simulations (30 param sets × 1 replicates)


FileNotFoundError: [Errno 2] No such file or directory: 'experiments\\mcmc-sampling\\data\\raw\\abm-data.pkl'

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA_PATH = Path("experiments/mcmc-sampling/data/raw/abm-data.csv")
PLOTS_DIR = Path("experiments/mcmc-sampling/out/plots/mcmc_sampling_plots")

data = data

params = {
    'R0': ('R0', 'R\u2080', 'steelblue'),
    'tau': ('tau','\u03C4 (transmission rate)', 'darkorange'),
    'rho': ('rho', '\u03C1 (initial infected)',  'mediumpurple'),
    'attack_rate': ('attack_rate', 'Attack Rate',                'firebrick'),
    'final_R': ('final_R', 'Final Recovered (R)',        'seagreen'),
    'near_threshold': ('near_threshold', 'Near Threshold (R\u2080\u22481)', 'goldenrod'),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (col, (key, label, color)) in zip(axes, params.items()):
    values = data[key].dropna()

    if key == 'near_threshold':
        counts = values.value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values, color=[color, 'lightgray'][:len(counts)],
               alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Not near (0)', 'Near (1)'], fontsize=10)
        pct = values.mean() * 100
        ax.set_title(f'{label}\n({pct:.1f}% near threshold)', fontsize=11, fontweight='bold')
    else:
        counts, bin_edges = np.histogram(values, bins=30)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        width = bin_edges[1] - bin_edges[0]
        ax.bar(bin_centers, counts, width=width * 0.85, color=color, alpha=0.85,
               edgecolor='white', linewidth=0.4)
        ax.axvline(values.mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean = {values.mean():.4f}')
        ax.legend(fontsize=9)
        ax.set_title(f'Distribution of {label}', fontsize=11, fontweight='bold')

    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('MCMC Posterior — Parameter & Outcome Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()

out = PLOTS_DIR / "parameter_bar_plots.png"
plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {out}")

NameError: name 'data' is not defined